In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.gold"

SHIPMENT = f"{SILVER}.fact_shipment_current"
DEPOT = f"{SILVER}.dim_depot"

TARGET = f"{GOLD}.gold_depot_performance"

In [0]:
depot_perf_df = (
    spark.table(SHIPMENT).alias("s")
    .join(
        spark.table(DEPOT).alias("d"),
        F.col("s.source_depot_id") == F.col("d.depot_id"),
        "left"
    )
    .groupBy(
        "d.depot_id",
        "d.depot_name"
    )
    .agg(
        F.count("*")
            .alias("total_shipments"),

        F.sum(
            F.when(
                F.col("current_status") == "DELIVERED",
                1
            ).otherwise(0)
        ).alias("delivered_shipments"),

        F.sum(
            F.when(
                F.col("delivery_delay_days") > 0,
                1
            ).otherwise(0)
        ).alias("delayed_shipments"),

        F.avg(
            "shipment_lead_time_days"
        ).alias("avg_lead_time_days")
    )
)

In [0]:
depot_perf_df = (
    depot_perf_df
    .withColumn(
        "on_time_delivery_pct",
        F.round(
            (
                (
                    F.col("delivered_shipments")
                    -
                    F.col("delayed_shipments")
                )
                /
                F.col("delivered_shipments")
            ) * 100,
            2
        )
    )
)

depot_perf_df = (
    depot_perf_df
    .withColumn(
        "on_time_delivery_pct",
        F.when(
            F.col("delivered_shipments") > 0,
            F.round(
                (
                    (
                        F.col("delivered_shipments")
                        -
                        F.col("delayed_shipments")
                    )
                    /
                    F.col("delivered_shipments")
                ) * 100,
                2
            )
        ).otherwise(0.0)
    )
)

In [0]:
(
    depot_perf_df.select(
        "depot_id",
        "depot_name",
        "total_shipments",
        "delivered_shipments",
        "delayed_shipments",
        "avg_lead_time_days",
        "on_time_delivery_pct"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET)
)

In [0]:
display(
    spark.table(TARGET)
)